# Project Config

In [3]:
from pathlib import Path
import subprocess

In [ ]:
# import os
# os.kill(os.getpid(), 9)

: 

: 

: 

In [1]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

Using device: cuda


In [9]:
def make_dir(dir_path: Path):
    dir_path.mkdir(parents=True, exist_ok=True)

def file_structure(assets_dir: Path):
    make_dir(assets_dir)

    input_dir = assets_dir / "raw"
    make_dir(input_dir)
    queue_dir = assets_dir / "queue"
    make_dir(queue_dir)
    history_dir = assets_dir / "processed"
    make_dir(history_dir)
    processed_csv_dir = history_dir / "csv"
    make_dir(processed_csv_dir)
    processed_json_dir = history_dir / "json"
    make_dir(processed_json_dir)

    return input_dir, queue_dir, history_dir
    
 
    # csv_dir = history_dir / "csv"
    # make_dir(csv_dir)

def clean_dir(dir_path: Path):
    for file in dir_path.glob("*"):
        if file.is_dir():
            clean_dir(file)
        else:
            file.unlink()
    dir_path.rmdir()


In [10]:
project_root = Path('__file__').parent
data_dir  = Path(project_root / "note_assets")

raw_dir, queue_dir, processed_dir = file_structure(data_dir)

#### *clear all files in the data set dir

In [ ]:
# clean_dir(data_dir)

# load data into input file

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download(
    "pad19tue/lecture-videos",
    output_dir=raw_dir,
    )
print("Path to dataset files:", path)
# move files to raw directory 

video_dir = raw_dir / "video"
print 
for mp4 in video_dir.glob("*.mp4"):
    mp4.rename(raw_dir / mp4.name)

100%|██████████| 5.07G/5.07G [00:35<00:00, 152MB/s] 

Extracting files...


Path to dataset files: note_assets/raw


# Video / Audio Processing 

In [6]:
def chunk_media(source: str, output_dir: str, seconds: int = 30, video_id: str = None):
    source = Path(source)
    output = Path(output_dir)
    pattern = output / f"{video_id}_%06d{source.suffix}"
    command = [
        "ffmpeg",
        "-y",
        "-i", str(source), #input file
        "-f", "segment", #format
        "-segment_time", str(seconds), #segment time
        "-reset_timestamps", "1", #reset timestamps
        "-c", "copy", #copy
        "-segment_list", str(output / f"{video_id}_segments.csv"),
        "-segment_list_type", "csv",
        str(pattern), #output pattern
    ]
    subprocess.run(command, check=True)

for file in raw_dir.glob("*.mp[34]"):
    video_id = file.stem
    chunk_media(file, queue_dir, video_id=video_id)
    file.rename(processed_dir / file.name)


# Audio to Text Processing

This section explores whether splitting the audio into shunks and transcribing it makes it faster than processing the whole audio

In [7]:
import whisper
import time
model = whisper.load_model("small", device= "cuda" if torch.cuda.is_available() else "cpu")
audio_data = []

100%|███████████████████████████████████████| 461M/461M [00:23<00:00, 21.0MiB/s]


# Transcribe whole file

In [9]:
import json

for file in processed_dir.glob("*.mp4"):
    if queue_dir / f"{file.stem}.json" in queue_dir.glob("*.json"):
        print(f"Skipping {file.stem} because it already exists")
        continue
    video_id = file.stem
    print(video_id)
    transcribed_text = model.transcribe(str(file), word_timestamps=False, fp16=device == "cuda")
    upload_json = {
        "video_id": video_id,
        "text": transcribed_text["text"],
        "segments": transcribed_text["segments"]
    }   
    
    with open(queue_dir / f"{video_id}.json", "w") as f:
            f.write(json.dumps(upload_json))




cognitive_robotics_high_res
team_dynamics_high_res
psychology_high_res
computer_vision_2_2_high_res
short_range_MIT
physics_high_res
ML_for_health_high_res
physics
cities_and_decarbonization_standard_res
sensory_systems_high_res
deep_learning_high_res
cryptocurrency_high_res
clinical_care_high_res
numerics_high_res
image_processing_high_res
phonetics_high_res
creating_breakthrough_products_MIT
reinforcement_learning_high_res
theory_of_computation_high_res
cities_and_climate_high_res
solar_resource_high_res


# Embedder - Embedding Gemma 2 by google


In [6]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer


print(f"Using device: {device}")

embedding_model = SentenceTransformer(
    "google/embeddinggemma-2",
    device=device,
    model_kwargs={"torch_dtype": torch.float32},
    config_kwargs={"audio_config": None},
)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Using device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/22.2k [00:00<?, ?B/s]

... (9 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

In [9]:
# video data preparation
import pandas as pd

video_metadata = []
for file in queue_dir.glob("*.csv"):
    l = f"{file.stem}.mp4"
    df = pd.read_csv(file, header=None, names=["name", "start", "end"])
    for index, row in df.iterrows():
        video_metadata.append({
            "original_file": l,
            "clip_id": index,
            "modality": "video",
            "clip_file_name": row["name"],
            "start": row["start"],
            "end": row["end"]
        })
    file.rename(processed_dir / "csv" / file.name)
print(pd.DataFrame(video_metadata).head(1))



                   original_file  clip_id modality  \
0  physics_high_res_segments.mp4        0    video   

                clip_file_name  start        end  
0  physics_high_res_000000.mp4    0.0  32.033333  


# Vector Databse

In [11]:
from pinecone import Pinecone, SchemaBuilder
from dotenv import load_dotenv
import os

load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))



### Create Index

In [12]:
schema = (
    SchemaBuilder()
    .add_dense_vector_field(name="embedding", dimension=768, metric="cosine")
    .build()
)

if not pc.indexes.exists("mm-vectors"):
    print("Creating index...")
    pc.indexes.create(name="mm-vectors",schema=schema)
    print("mm-vectors index created")
mm_index = pc.Index("mm-vectors")

### pass vido file into embedder and insert into database vector

In [ ]:
for index, video in enumerate(video_metadata):
    file_path = queue_dir / video["file_name"]
    video_id = video["video_id"]
    clip_id = video["clip_id"]
    id = f"{video_id}_{clip_id}"
    if not file_path.exists():
        continue
    
    video_embedding = model.encode(
        str(file_path),
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=device,
        show_progress_bar=True,
        processing_kwargs={
            "fps": 1,
            "max_frames": 32,
            "add_timestamps": True,
            "max_soft_tokens": 280,
            "overflow_strategy": "uniform",
        }
    ).reshape(-1)
    video_metadata[index]["embedding"] = video_embedding.tolist()
    

In [ ]:
for video in video_metadata:
    mm_index.documents.upsert(
            namespace="videos",
            documents=[
                {
                    "_id": video["clip_file_name"],
                    "embedding": video["embedding"],
                    "clip_id": video["clip_id"],
                    "original_file": video["original_file"],
                    "modality": "video",
                    "start": float(video["start"]),
                    "end": float(video["end"]),
                }
            ],
        )

In [7]:
import re

def clean_text(text: str):
        return re.sub(r"\s+", " ", text).strip()

def split_text(text:str, count_tokens, max_tokens: int):
    if count_tokens(text) <= max_tokens:
        return [text]
        
    middle = len(text) // 2
    spaces = [m.start() for m in re.finditer(r"\s+", text)]
    cut = min(spaces, key=lambda x: abs(x - middle)) if spaces else middle

    if not 0 < cut < len(text):
        raise ValueError("Token budget is too small for one character.")
        
    return (split_text(text[:cut].strip(), count_tokens, max_tokens)
                    + split_text(text[cut:].strip(), count_tokens, max_tokens))
     
def build_chunks(segments, id, count_tokens, budget:int = 220, overlap: int = 40):
    if not 0 <= overlap < budget:
        raise ValueError("Overlap must be between 0 and budget.")
        
    units = []
    for i, segment in enumerate(segments):
        text = clean_text(segment["text"])
        if not text:
            continue

        start = float(segment["start"])
        end = float(segment["end"])
        
        for piece in split_text(text, count_tokens, budget):
            units.append({
                "text": piece,
                "start": start,
                "end": end,
                "segment_id": segment.get("id", i),
            })

    chunks = []
    i = 0
    chunk_id = 0
    while i < len(units):
        j = i
        text = ""
        while j< len(units):
            combined_text = (text+ " " + units[j]["text"]).strip()
            if count_tokens(combined_text) <= budget:
                text = combined_text
                j += 1
            else:
                break

        if i == j:
            raise ValueError("Token budget is too small for one character.")
            
        selected_chunk = units[i:j]
        chunks.append({
            "id": f"{id}_{chunk_id}",
            "original_file": f"{id}.json",
            "modality": f"audio_text_chunks",
            "start": selected_chunk[0]["start"],
            "end": selected_chunk[-1]["end"],
            "text": text,
        })
        chunk_id += 1
        if j == len(units):
            break
            
        temp = j
        while temp > i + 1:
            suffix = " ".join(u["text"] for u in units[temp - 1:j])
            if count_tokens(suffix) > overlap:
                break
            temp -= 1
        i = temp

    return chunks
    
tokenizer = embedding_model.tokenizer

def count_tokens(text: str):
    return len(tokenizer.encode(text, add_special_tokens=False, truncation=True))

In [13]:
import json
original_text = []
chunks = []
for file in queue_dir.glob("*.json"):
    meta_data = []
    with open(file, "r") as f:
        text = f.read()
    text = json.loads(text)
    video_id = text["video_id"]
    original_text.append({
        "id": f"{video_id}.json",
        "modality": "audio_text_full",
        "text": text["text"]
    })

    chunks += build_chunks(text["segments"], video_id, count_tokens, 500, 100)
    file.rename(processed_dir / "json" / file.name)


In [15]:
for index, chunk in enumerate(chunks):
    text_embedding = embedding_model.encode(
        chunk["text"],
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=device,
    ).reshape(-1)
    
    chunks[index]["embedding"] = text_embedding.tolist()
    

In [16]:
for chunk in chunks:
    mm_index.documents.upsert(
        namespace="text",
        documents=[
            {
                "_id": chunk["id"],
                "embedding": chunk["embedding"],
                "original_file": chunk["original_file"],
                "start": float(chunk["start"]),
                "end": float(chunk["end"]),
                "modality": chunk["modality"],
                "text": chunk["text"],
            }
        ]
    )